# GTJA191 Demo — Regenerate Factors & Backtest

Runnable in **JupyterLab**. Pipeline:

1. Setup & import local `gtja191` package
2. Load market data (AkShare or synthetic)
3. Regenerate alpha factors → Parquet store
4. IC / quintile diagnostics
5. **Visualizations** (prices, IC, heatmaps, quintiles)

> v0.1 implements **22 factors**: alpha 001–020, 171, 191.

In [1]:
# --- Cell 1: Setup paths & install package (run once) ---
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "gtja191").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # if launched from notebooks/

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Optional: pip install -e in terminal instead
# !pip install -e "{PROJECT_ROOT}"

In [2]:
# --- Cell 2: Imports & configuration ---
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

from gtja191 import GTJA191Engine, N_FACTORS, load_sample_universe
from gtja191.factors import list_implemented
from gtja191.pipeline import FactorRegenerator
from gtja191.backtest import FactorAnalyzer, forward_return, load_backtest_config
from gtja191.data.synthetic import make_synthetic_panel
from gtja191.data import AkShareLoader
from gtja191.viz import (
    make_demo_dashboard,
    make_quintile_figure,
    plot_factor_return_scatter,
)

DATA_DIR = PROJECT_ROOT / "data"
MARKET_DIR = DATA_DIR / "market"
FACTOR_DIR = DATA_DIR / "factors"
FIG_DIR = DATA_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

START = "2023-01-01"
END = "2024-12-31"
USE_AKSHARE = True  # set False to use synthetic data only
SYMBOLS = load_sample_universe()  # 5 tickers (IC needs >=5 names per day)
SHOWCASE_FACTORS = [1, 2, 171, 191]

print(f"GTJA191 target factors: {N_FACTORS}")
print(f"Implemented now: {list_implemented()}")
print(f"Universe: {SYMBOLS}")

GTJA191 target factors: 191
Implemented now: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 171, 191]
Universe: ['600519', '000001', '601318', '600036', '000858']


In [3]:
# --- Cell 3: Load market data ---
panel = None
if USE_AKSHARE:
    try:
        loader = AkShareLoader(cache_dir=DATA_DIR / "akshare_cache")
        panel = loader.load(SYMBOLS, START, END, adjust="qfq")
        print("Loaded from AkShare:", {k: v.shape for k, v in panel.items()})
    except Exception as e:
        print(f"AkShare failed ({e}); falling back to synthetic data.")

if panel is None:
    panel = make_synthetic_panel(SYMBOLS, START, END)
    print("Using synthetic panel:", {k: v.shape for k, v in panel.items()})

panel["close"].tail()

AkShare failed (No module named 'akshare'); falling back to synthetic data.
Using synthetic panel: {'open': (522, 5), 'high': (522, 5), 'low': (522, 5), 'close': (522, 5), 'volume': (522, 5), 'amount': (522, 5)}


,600519,000001,601318,600036,000858
2024-12-25,11.040060,8.106980,10.343905,3.060252,24.117145
2024-12-26,11.245711,7.957617,10.181435,3.125034,23.654800
2024-12-27,11.623926,8.118192,9.955541,3.175569,23.573804
2024-12-30,11.406263,8.236527,9.774969,3.171108,23.979027
2024-12-31,11.211148,8.257462,9.656475,3.143138,23.360861


In [4]:
# --- Cell 4: Regenerate factors & persist to Parquet ---
regen = FactorRegenerator(output_dir=FACTOR_DIR, cache_dir=MARKET_DIR)
manifest = regen.run(
    universe=SYMBOLS,
    start=START,
    end=END,
    factors="all",
    panel=panel,
)
manifest

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [5]:
# --- Cell 5: Quick compute in-memory (no store) ---
engine = GTJA191Engine(panel)
factors = engine.compute(SHOWCASE_FACTORS, return_format="dict")
for fid, df in factors.items():
    print(f"alpha_{fid:03d} last row:\n", df.iloc[-1].dropna())

alpha_001 last row:
 600519    3.908428e-01
000001   -1.462545e-01
601318   -1.264912e-15
600036    3.223292e-01
000858    4.285714e-01
Name: 2024-12-31 00:00:00, dtype: float64
alpha_002 last row:
 600519    0.885490
000001   -0.186778
601318   -0.552709
600036    0.085581
000858   -0.624494
Name: 2024-12-31 00:00:00, dtype: float64
alpha_171 last row:
 600519   -0.279559
000001   -0.513283
601318   -0.583870
600036   -0.364171
000858   -3.621751
Name: 2024-12-31 00:00:00, dtype: float64
alpha_191 last row:
 600519    6.757295e-01
000001   -9.090909e-02
601318    2.773501e-01
600036    8.618925e-16
000858   -3.175537e-01
Name: 2024-12-31 00:00:00, dtype: float64


In [6]:
# --- Cell 6: IC & quintile backtest ---
cfg = load_backtest_config()
cfg["n_quintiles"] = 3
analyzer = FactorAnalyzer(factor_store=FACTOR_DIR, market_store=MARKET_DIR, config=cfg)

reports = {}
for fid in SHOWCASE_FACTORS:
    try:
        rep = analyzer.evaluate(fid, start=START, end=END)
        reports[fid] = rep
        print(
            f"alpha_{fid:03d} | IC_mean={rep.ic.ic_mean:.4f} IR={rep.ic.ir:.4f} "
            f"hit={rep.ic.ic_hit_rate:.2%} quintile_spread={rep.quintile_spread:.4f}"
        )
        print("  quintiles:", rep.quintiles.to_dict())
    except FileNotFoundError:
        print(f"alpha_{fid:03d} not in store — run regeneration cell first")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [7]:
# --- Cell 7: Forward return sanity check (no lookahead) ---
open_ = panel["open"]
close = panel["close"]
h = int(cfg.get("forward_return_horizon", 5))
fwd = forward_return(open_, close, horizon=h)
print(f"Forward return horizon H={h} (buy T+1 open, sell T+H close)")
fwd.iloc[-10:].mean(axis=1)

Forward return horizon H=5 (buy T+1 open, sell T+H close)


2024-12-18    0.021746
2024-12-19    0.013659
2024-12-20    0.011785
2024-12-23    0.009179
2024-12-24   -0.011973
2024-12-25         NaN
2024-12-26         NaN
2024-12-27         NaN
2024-12-30         NaN
2024-12-31         NaN
Freq: B, dtype: float64

## Visualizations

Charts below summarize prices, factor heatmaps, daily/cumulative IC, and quintile returns.

In [8]:
# --- Cell 8: Main results dashboard ---
if reports:
    dashboard_path = FIG_DIR / "gtja191_dashboard.png"
    make_demo_dashboard(
        panel,
        factors,
        reports,
        fwd_ret=fwd,
        save_path=dashboard_path,
        show=True,
    )
    print(f"Saved: {dashboard_path}")
else:
    print("No reports — run cells 4–6 first.")

No reports — run cells 4–6 first.


In [9]:
# --- Cell 9: Quintile bars + factor-return scatter ---
if reports:
    quintile_path = FIG_DIR / "gtja191_quintiles.png"
    make_quintile_figure(reports, save_path=quintile_path, show=True)
    print(f"Saved: {quintile_path}")

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
    axes = axes.ravel()
    for ax, fid in zip(axes, SHOWCASE_FACTORS):
        if fid in factors:
            plot_factor_return_scatter(
                factors[fid], fwd, ax=ax, title=f"alpha_{fid:03d} vs fwd return"
            )
    scatter_path = FIG_DIR / "gtja191_scatter.png"
    fig.suptitle("Factor vs forward return (recent dates)", fontweight="bold")
    fig.savefig(scatter_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {scatter_path}")
else:
    print("No reports — run cells 4–6 first.")

No reports — run cells 4–6 first.
